[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabriceyhc/emerging-substances-la/blob/main/notebooks/emergence_validation.ipynb)

# Emergence validation: annotated alarm review

Reviews every substance that at least one of five detection methods has ever alarmed on, annotated against the same raw death-count line.

**Data sources** (regenerate with the commands shown if stale):

| file | command | grain |
|---|---|---|
| `results/trends/rising_substances.csv` | `emerging trends rank` | current EB05++TS score per substance, used only to order Part 1's cells and Part 2's grid |
| `results/trends/emergence_by_quarter.csv` | `emerging trends emergence-table` | substance x quarter, raw death counts |
| `results/trends/total_deaths_by_quarter.csv` | `emerging trends emergence-table` | all-OD deaths per quarter, the denominator behind each chart's second (share) line |
| `data/raw/ground_truth/known_emergences.csv` | hand-curated (see `emerging/validation/ground_truth.py`) | the project's actual reference intervals -- seeds each Part 1 cell's `spans=[...]` where a substance already has one |
| `results/polysubstance/profile.csv` | `emerging polysubstance profile` | trailing-4-quarter co-occurrence role (independent / principal driver / adulterant-marker / ...) -- seeds each Part 1 cell's note where a substance has enough recent cases to profile |
| `results/trends/emergence_by_year.csv` | `emerging trends emergence-table` | substance x year, raw death counts (the requested deliverable table; not plotted here, quarterly is used instead so alarm markers land on the right x-position) |
| `results/benchmark/method_timeline.csv` | `emerging benchmark method-timeline` | substance x as-of-quarter x method, score + alarm flag |

**The six methods**, an escalating ladder of sophistication (`docs/GPS_V2_DESIGN.md`), not six arbitrary picks:

| label | what it is | id in `emerging/validation/benchmark.py` |
|---|---|---|
| `n/E` | raw ratio, no shrinkage at all -- plotted for context but **excluded** from "has this substance ever fired" below: with no shrinkage at all it alarms on nearly every new arrival (872 of the sweep's ~3,700 substance-quarters), so using it as a filter would defeat the point of picking substances worth reviewing | `ratio` |
| `EB05` | classic single-gate EB05, raw counts | `eb05` |
| `EB05+` | dual-gated EB05 (share up **and** own-count up), raw counts | `eb05-dual` |
| `TreeScan` | substance-leaf recurrence interval, solo | `treescan` |
| `NB-Trend` | log-linear Poisson trend, Wald z of the fitted slope | `nb-trend` |
| `EB05++TS` | **the deployed detector** -- weighted + role-discounted + spatial-fused dual-gated EB05, TreeScan-vetoed at RI 10 | not a single benchmark.py id; read from `trends alarms`'s own cache (`eb05_sweep.csv`), the one reading here expensive enough that `alarms` already pays that cost |

Every method's raw score is on an incomparable scale to the others (a ratio, a posterior percentile, a recurrence interval, a Wald z) -- this notebook never plots them against each other, only *when each one alarmed*, as markers on the one line that is comparable: the substance's own raw death count.

**Two more datapoints beyond the count line itself**, both already used in the actual `known_emergences.csv` reasoning rather than new for this notebook:

- **Share of all OD deaths** (blue dashed line, right axis) -- a raw count conflates a substance's own trend with the county's total overdose volume changing underneath it. This is exactly the rate `known_emergences.csv`'s notes read for the established substances (Fentanyl: "1.64% (2016Q1) -> 6.56% (2016Q2)"); verified here to the decimal against that file's own numbers.
- **Polysubstance role** (each cell's note, where a substance has enough recent cases) -- `independent` / `principal driver` / `adulterant-marker` / etc. from `emerging polysubstance profile`'s co-occurrence and cause-line-position analysis. This is the actual evidence that separated real negatives already in the ground truth (Lidocaine, Levamisole -- named last, almost always with fentanyl, i.e. a cutting agent) from real emergences, independent of how sharp the count curve looks.

**Part 1's cells are generated, not hand-written** -- one substance, one cell, each seeded with that substance's `known_emergences.csv` interval if it has one. Editing a cell's `spans=[...]` list and re-running it is the intended way to try out a candidate window visually; once you're satisfied a window is right, the place that opinion actually belongs is a row in `known_emergences.csv` itself (`docs/findings/ground_truth.md` documents the process), not this notebook -- regenerating Part 1 later (when a newly-fired substance needs adding) re-seeds every cell from that file and will overwrite in-notebook edits that were never promoted there.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

%matplotlib inline

# Locally (VS Code, JupyterLab) this notebook is assumed to stay at
# notebooks/emergence_validation.ipynb, one level under the repo root, and
# nothing below runs. Colab starts with an empty runtime with no checkout of
# this repo at all, so clone it there -- the three CSVs this notebook reads
# are aggregate counts already committed to the repo (`emerging/paths.py`'s
# rule: no case numbers, no coordinates in `results/`), so cloning is safe.
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import os
    REPO = Path("/content/emerging-substances-la")
    if REPO.exists() and not (REPO / ".git").exists():
        # A previous run's clone was interrupted (e.g. it hung and the cell
        # was stopped) and left a partial, non-git directory behind -- retry
        # from scratch rather than silently reusing a broken checkout.
        !rm -rf {REPO}
    if not REPO.exists():
        # Fail fast instead of hanging: with no credential helper configured,
        # a clone that unexpectedly needs auth would otherwise sit forever on
        # a username prompt Colab's `!` shell can never answer.
        os.environ["GIT_TERMINAL_PROMPT"] = "0"
        !git clone -q https://github.com/fabriceyhc/emerging-substances-la.git {REPO}
    else:
        # A Colab runtime can persist in the background across cell re-runs
        # -- and even across closing and reopening the notebook -- so a
        # checkout from an earlier run can still be sitting there. Force an
        # exact match to origin/main (fetch + hard reset, not `pull`) rather
        # than trust it: this is a read-only mirror, nothing here is ever
        # meant to diverge from the remote, so there's no local state worth
        # a merge -- and a plain `pull` can itself fail quietly (a hung
        # fetch, a `!` shell error that doesn't raise in Python) leaving a
        # stale checkout with no sign anything went wrong.
        !git -C {REPO} fetch -q origin main && git -C {REPO} reset -q --hard origin/main
    assert (REPO / ".git").exists(), "clone failed -- see the git output above"
    # The fetch+reset above can itself fail silently (see comment) -- check
    # for a file this notebook actually needs so a stale checkout raises a
    # clear, actionable message here, not a bare FileNotFoundError several
    # cells later with no indication the repo sync was the problem.
    _sentinel = REPO / "results" / "trends" / "total_deaths_by_quarter.csv"
    assert _sentinel.exists(), (
        f"{_sentinel} is missing after clone/sync -- {REPO} is stale and "
        f"didn't update. Run `!rm -rf {REPO}` in a new cell, then re-run "
        "this cell to force a fresh clone.")
    ROOT = REPO
else:
    ROOT = Path("..")
RESULTS = ROOT / "results"

In [ ]:
timeline = pd.read_csv(RESULTS / "benchmark" / "method_timeline.csv", parse_dates=["as_of"])
quarterly = pd.read_csv(RESULTS / "trends" / "emergence_by_quarter.csv", index_col=0)
quarterly.columns = pd.to_datetime(quarterly.columns)
ranking = pd.read_csv(RESULTS / "trends" / "rising_substances.csv", index_col=0)["eb05"]

# Share of *all* OD deaths that quarter, not just this substance's own count
# -- same denominator `load_quarterly` calls `denom` (the plain cohort
# count, no weighting), which is what `known_emergences.csv`'s notes
# actually read a rate off (verified to the decimal in the build script).
total_deaths = pd.read_csv(RESULTS / "trends" / "total_deaths_by_quarter.csv",
                           index_col=0)["n_total"]
total_deaths.index = pd.to_datetime(total_deaths.index)
share = 100 * quarterly.div(total_deaths.reindex(quarterly.columns), axis=1)

# "Fired" = alarmed under at least one of the four non-ensemble methods or
# the deployed detector -- n/E excluded (see the table above). This is a
# strict superset of `alarm_history.csv`'s substances (which is EB05++TS
# alone): TreeScan and NB-Trend in particular each flag substances the
# deployed detector's own dual gate + veto never do, and part of this
# review's point is to see where the methods disagree, not just where the
# deployed one already agrees with itself.
ALARM_METHODS = ["EB05", "EB05+", "TreeScan", "NB-Trend", "EB05++TS"]
ever_alarmed = timeline.loc[timeline["method"].isin(ALARM_METHODS) & timeline["alarm"],
                           "substance"].unique()

# Ordered by current EB05++TS score, descending -- same convention as Part
# 2's grid and `geo export`'s one-hot columns, so all three read substances
# in the same priority order. A few names here alarmed only under TreeScan
# or NB-Trend and never scored highly on EB05++TS itself, so they sit near
# the bottom of this list despite being exactly the disagreements worth
# reviewing -- that ordering choice, not a bug.
FIRED = (ranking.reindex(ever_alarmed)
         .sort_values(ascending=False, na_position="last").index.tolist())
print(f"{len(FIRED)} substances have ever fired one of {ALARM_METHODS}:")
print(FIRED)

In [ ]:
gt = pd.read_csv(ROOT / "data" / "raw" / "ground_truth" / "known_emergences.csv")


def _quarter(s):
    """'2020Q2' -> Timestamp('2020-04-01'); blank/NaN -> None (open-ended)."""
    return None if pd.isna(s) else pd.Period(s, freq="Q").start_time


# One row per substance in the CSV today, so each seed is a single-span list
# -- `plot_substance` itself takes a *list* of (start, end) pairs, because a
# substance can have more than one real episode (Acetyl fentanyl's notes
# flag a possible second one the current single-row schema can't hold yet).
KNOWN_SPANS = {r["substance"]: [(_quarter(r["interval_start"]), _quarter(r["interval_end"]))]
               for _, r in gt[~gt["no_emergence"]].iterrows()}
# Documented negatives: an interval is in the CSV only as a parsing
# placeholder (see known_emergences.csv's own docstring) and would mislead
# if shaded here as though it were a real reference window.
NEGATIVE_CASES = set(gt.loc[gt["no_emergence"], "substance"])

## Part 1 -- annotated review: substances that fired

One cell per substance below -- any substance at least one of EB05, EB05+, TreeScan, NB-Trend or EB05++TS has ever alarmed on, not only the deployed detector's own list (35 today; see the count printed above). The black line is the raw quarterly death count (`emergence_by_quarter.csv`); each colored row of markers above it is one method, placed at the as-of quarters where *that method's own* alarm condition was true (`method_timeline.csv`'s `alarm` column) -- not at a shared y-value tied to the line, so a quiet quarter's markers don't collide with the line itself. `n/E` still plots as context (grey dots) even though it doesn't gate which substances appear here.

The grey shaded band(s), where present, are `spans` -- reference interval(s) either seeded from `known_emergences.csv` or left for you to fill in. Edit the list in a cell and re-run just that cell to try a different window; `end=None` means open-ended (shades through to the last available quarter).

In [ ]:
METHOD_STYLE = {
    "n/E":      dict(marker="o", color="#b8b7b2"),
    "EB05":     dict(marker="s", color="#2a78d6"),
    "EB05+":    dict(marker="^", color="#1baf7a"),
    "TreeScan": dict(marker="D", color="#eda100"),
    "NB-Trend": dict(marker="v", color="#7a4fd1"),
    "EB05++TS": dict(marker="*", color="#eb6834"),
}
METHOD_ORDER = list(METHOD_STYLE)


def plot_substance(substance, spans=None):
    """`spans`: a list of (start, end) reference intervals to shade, each
    a date-like string/Timestamp; `end=None` means open-ended, shaded
    through to the last available quarter."""
    series = quarterly.loc[substance]
    fig, ax = plt.subplots(figsize=(11, 4.5))

    for i, (start, end) in enumerate(spans or []):
        ax.axvspan(pd.Timestamp(start),
                  pd.Timestamp(end) if end is not None else series.index.max(),
                  color="#b8b7b2", alpha=0.25, zorder=0,
                  label="reference interval" if i == 0 else None)

    ax.plot(series.index, series.values, color="#0b0b0b", linewidth=1.6, zorder=2)
    ax.fill_between(series.index, series.values, color="#0b0b0b", alpha=0.05, zorder=1)

    # Share of *all* OD deaths, right axis -- a raw count conflates a
    # substance's own trend with the county's total overdose volume moving
    # underneath it; share is the rate `known_emergences.csv`'s own notes
    # read a verdict off for the established substances.
    ax2 = ax.twinx()
    share_series = share.loc[substance]
    ax2.plot(share_series.index, share_series.values, color="#2a78d6",
            linewidth=1.2, linestyle="--", alpha=0.8, zorder=2,
            label="% of all OD deaths")
    ax2.set_ylabel("% of all OD deaths", fontsize=8.5, color="#2a78d6")
    ax2.tick_params(axis="y", labelsize=8, colors="#2a78d6")
    ax2.spines["top"].set_visible(False)
    ax2.spines["right"].set_color("#2a78d6")
    ax2.set_ylim(0, max(share_series.max(), 0.1) * 1.4)

    sub = timeline[timeline["substance"] == substance]
    ymax = max(series.max(), 1)
    for i, method in enumerate(METHOD_ORDER):
        fired_q = sub[(sub["method"] == method) & sub["alarm"]]
        if not len(fired_q):
            continue
        style = METHOD_STYLE[method]
        y = ymax * (1.06 + 0.05 * i)
        ax.scatter(fired_q["as_of"], [y] * len(fired_q), label=method,
                   marker=style["marker"], color=style["color"],
                   edgecolors="white", linewidths=0.6, s=45, zorder=3)

    ax.set_title(f"{substance} -- quarterly deaths, six methods' alarm quarters",
                loc="left", fontsize=12)
    ax.set_ylim(0, ymax * 1.4)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    h1, l1 = ax.get_legend_handles_labels()
    h2, l2 = ax2.get_legend_handles_labels()
    ax.legend(h1 + h2, l1 + l2, frameon=False, ncol=8, loc="upper left",
             bbox_to_anchor=(0, -0.08), fontsize=8.5)
    fig.tight_layout()
    plt.show()
    plt.close(fig)

### PCP

*`known_emergences.csv`: 2020-04-01 to ongoing*

*polysubstance (trailing 4q, n=137): **secondary/incidental** -- alone 0%, named last 81%, with fentanyl 51%*

In [ ]:
plot_substance("PCP", spans=[
    ("2020-04-01", None),
])

### Cocaine

*not yet in `known_emergences.csv`*

*polysubstance (trailing 4q, n=455): **principal driver** -- alone 26%, named last 31%, with fentanyl 49%*

In [ ]:
plot_substance("Cocaine", spans=[
    # ("YYYY-MM-DD", "YYYY-MM-DD"),  # add a candidate window, or leave empty
])

### Alprazolam

*documented negative case in `known_emergences.csv` (`no_emergence=True`) -- no reference interval to shade*

*polysubstance (trailing 4q, n=110): **adulterant/marker** -- alone 1%, named last 46%, with fentanyl 75%*

In [ ]:
plot_substance("Alprazolam", spans=[
    # documented negative case -- see the note above; no interval to seed
])

### Carfentanil

*`known_emergences.csv`: 2024-01-01 to ongoing*

*polysubstance (trailing 4q, n=10): **principal driver** -- alone 20%, named last 12%, with fentanyl 60%*

In [ ]:
plot_substance("Carfentanil", spans=[
    ("2024-01-01", None),
])

### Methamphetamine

*not yet in `known_emergences.csv`*

*polysubstance (trailing 4q, n=1345): **independent** -- alone 37%, named last 66%, with fentanyl 49%*

In [ ]:
plot_substance("Methamphetamine", spans=[
    # ("YYYY-MM-DD", "YYYY-MM-DD"),  # add a candidate window, or leave empty
])

### Oxycodone

*not yet in `known_emergences.csv`*

*polysubstance (trailing 4q, n=34): **co-intoxicant** -- alone 9%, named last 36%, with fentanyl 38%*

In [ ]:
plot_substance("Oxycodone", spans=[
    # ("YYYY-MM-DD", "YYYY-MM-DD"),  # add a candidate window, or leave empty
])

### Amphetamine

*not yet in `known_emergences.csv`*

*polysubstance (trailing 4q, n=25): **independent** -- alone 40%, named last 73%, with fentanyl 48%*

In [ ]:
plot_substance("Amphetamine", spans=[
    # ("YYYY-MM-DD", "YYYY-MM-DD"),  # add a candidate window, or leave empty
])

### 1,1-Difluoroethane

*not yet in `known_emergences.csv`*

*polysubstance (trailing 4q, n=8): **co-intoxicant** -- alone 0%, named last 25%, with fentanyl 50%*

In [ ]:
plot_substance("1,1-Difluoroethane", spans=[
    # ("YYYY-MM-DD", "YYYY-MM-DD"),  # add a candidate window, or leave empty
])

### Ketamine

*`known_emergences.csv`: 2022-07-01 to ongoing*

*polysubstance (trailing 4q, n=19): **co-intoxicant** -- alone 5%, named last 50%, with fentanyl 63%*

In [ ]:
plot_substance("Ketamine", spans=[
    ("2022-07-01", None),
])

### Morphine

*documented negative case in `known_emergences.csv` (`no_emergence=True`) -- no reference interval to shade*

*polysubstance (trailing 4q, n=29): **co-intoxicant** -- alone 3%, named last 39%, with fentanyl 34%*

In [ ]:
plot_substance("Morphine", spans=[
    # documented negative case -- see the note above; no interval to seed
])

### Chlordiazepoxide

*not yet in `known_emergences.csv`*

*polysubstance (trailing 4q, n=6): **secondary/incidental** -- alone 0%, named last 40%, with fentanyl 50%*

In [ ]:
plot_substance("Chlordiazepoxide", spans=[
    # ("YYYY-MM-DD", "YYYY-MM-DD"),  # add a candidate window, or leave empty
])

### Fentanyl

*`known_emergences.csv`: 2016-04-01 to 2023-04-01*

*polysubstance (trailing 4q, n=1101): **principal driver** -- alone 15%, named last 9%, with fentanyl 0%*

In [ ]:
plot_substance("Fentanyl", spans=[
    ("2016-04-01", '2023-04-01'),
])

### Mitragynine

*`known_emergences.csv`: 2022-04-01 to ongoing*

*polysubstance (trailing 4q, n=18): **co-intoxicant** -- alone 0%, named last 43%, with fentanyl 61%*

In [ ]:
plot_substance("Mitragynine", spans=[
    ("2022-04-01", None),
])

### Codeine

*`known_emergences.csv`: 2023-10-01 to ongoing*

*polysubstance (trailing 4q, n=16): **secondary/incidental** -- alone 0%, named last 44%, with fentanyl 25%*

In [ ]:
plot_substance("Codeine", spans=[
    ("2023-10-01", None),
])

### Quetiapine

*not yet in `known_emergences.csv`*

*polysubstance (trailing 4q, n=10): **independent** -- alone 30%, named last 57%, with fentanyl 10%*

In [ ]:
plot_substance("Quetiapine", spans=[
    # ("YYYY-MM-DD", "YYYY-MM-DD"),  # add a candidate window, or leave empty
])

### Acetyl fentanyl

*`known_emergences.csv`: 2022-01-01 to 2024-04-01*

*polysubstance (trailing 4q, n=28): **co-intoxicant** -- alone 0%, named last 18%, with fentanyl 100%*

In [ ]:
plot_substance("Acetyl fentanyl", spans=[
    ("2022-01-01", '2024-04-01'),
])

### Buprenorphine

*not yet in `known_emergences.csv`*

*polysubstance (trailing 4q, n=4): **principal driver** -- alone 0%, named last 50%, with fentanyl 0%*

In [ ]:
plot_substance("Buprenorphine", spans=[
    # ("YYYY-MM-DD", "YYYY-MM-DD"),  # add a candidate window, or leave empty
])

### Clonazepam

*`known_emergences.csv`: 2019-07-01 to 2022-07-01*

*polysubstance (trailing 4q, n=11): **secondary/incidental** -- alone 0%, named last 67%, with fentanyl 64%*

In [ ]:
plot_substance("Clonazepam", spans=[
    ("2019-07-01", '2022-07-01'),
])

### Gabapentin

*not yet in `known_emergences.csv`*

*polysubstance: fewer than 4 cases in the trailing 4 quarters -- no current profile*

In [ ]:
plot_substance("Gabapentin", spans=[
    # ("YYYY-MM-DD", "YYYY-MM-DD"),  # add a candidate window, or leave empty
])

### Lidocaine

*documented negative case in `known_emergences.csv` (`no_emergence=True`) -- no reference interval to shade*

*polysubstance (trailing 4q, n=17): **adulterant/marker** -- alone 0%, named last 82%, with fentanyl 100%*

In [ ]:
plot_substance("Lidocaine", spans=[
    # documented negative case -- see the note above; no interval to seed
])

### Citalopram

*documented negative case in `known_emergences.csv` (`no_emergence=True`) -- no reference interval to shade*

*polysubstance (trailing 4q, n=5): **principal driver** -- alone 0%, named last 20%, with fentanyl 20%*

In [ ]:
plot_substance("Citalopram", spans=[
    # documented negative case -- see the note above; no interval to seed
])

### Mirtazapine

*documented negative case in `known_emergences.csv` (`no_emergence=True`) -- no reference interval to shade*

*polysubstance: fewer than 4 cases in the trailing 4 quarters -- no current profile*

In [ ]:
plot_substance("Mirtazapine", spans=[
    # documented negative case -- see the note above; no interval to seed
])

### Bromazolam

*`known_emergences.csv`: 2023-07-01 to 2025-01-01*

*polysubstance (trailing 4q, n=7): **principal driver** -- alone 0%, named last 29%, with fentanyl 14%*

In [ ]:
plot_substance("Bromazolam", spans=[
    ("2023-07-01", '2025-01-01'),
])

### Hydromorphone

*documented negative case in `known_emergences.csv` (`no_emergence=True`) -- no reference interval to shade*

*polysubstance (trailing 4q, n=4): **co-intoxicant** -- alone 0%, named last 25%, with fentanyl 0%*

In [ ]:
plot_substance("Hydromorphone", spans=[
    # documented negative case -- see the note above; no interval to seed
])

### Xylazine

*`known_emergences.csv`: 2023-07-01 to 2025-01-01*

*polysubstance: fewer than 4 cases in the trailing 4 quarters -- no current profile*

In [ ]:
plot_substance("Xylazine", spans=[
    ("2023-07-01", '2025-01-01'),
])

### Flualprazolam

*`known_emergences.csv`: 2020-01-01 to 2021-01-01*

*polysubstance: fewer than 4 cases in the trailing 4 quarters -- no current profile*

In [ ]:
plot_substance("Flualprazolam", spans=[
    ("2020-01-01", '2021-01-01'),
])

### Levamisole

*documented negative case in `known_emergences.csv` (`no_emergence=True`) -- no reference interval to shade*

*polysubstance: fewer than 4 cases in the trailing 4 quarters -- no current profile*

In [ ]:
plot_substance("Levamisole", spans=[
    # documented negative case -- see the note above; no interval to seed
])

### Etizolam

*`known_emergences.csv`: 2019-10-01 to 2021-10-01*

*polysubstance: fewer than 4 cases in the trailing 4 quarters -- no current profile*

In [ ]:
plot_substance("Etizolam", spans=[
    ("2019-10-01", '2021-10-01'),
])

### Methocarbamol

*not yet in `known_emergences.csv`*

*polysubstance: fewer than 4 cases in the trailing 4 quarters -- no current profile*

In [ ]:
plot_substance("Methocarbamol", spans=[
    # ("YYYY-MM-DD", "YYYY-MM-DD"),  # add a candidate window, or leave empty
])

### Diphenhydramine

*not yet in `known_emergences.csv`*

*polysubstance (trailing 4q, n=6): **secondary/incidental** -- alone 0%, named last 50%, with fentanyl 50%*

In [ ]:
plot_substance("Diphenhydramine", spans=[
    # ("YYYY-MM-DD", "YYYY-MM-DD"),  # add a candidate window, or leave empty
])

### Benzylfentanyl

*`known_emergences.csv`: 2024-04-01 to 2024-07-01*

*polysubstance: fewer than 4 cases in the trailing 4 quarters -- no current profile*

In [ ]:
plot_substance("Benzylfentanyl", spans=[
    ("2024-04-01", '2024-07-01'),
])

### Hydroxyzine

*not yet in `known_emergences.csv`*

*polysubstance: fewer than 4 cases in the trailing 4 quarters -- no current profile*

In [ ]:
plot_substance("Hydroxyzine", spans=[
    # ("YYYY-MM-DD", "YYYY-MM-DD"),  # add a candidate window, or leave empty
])

### Ephedrine

*`known_emergences.csv`: 2024-01-01 to 2024-10-01*

*polysubstance: fewer than 4 cases in the trailing 4 quarters -- no current profile*

In [ ]:
plot_substance("Ephedrine", spans=[
    ("2024-01-01", '2024-10-01'),
])

### Temazepam

*not yet in `known_emergences.csv`*

*polysubstance: fewer than 4 cases in the trailing 4 quarters -- no current profile*

In [ ]:
plot_substance("Temazepam", spans=[
    # ("YYYY-MM-DD", "YYYY-MM-DD"),  # add a candidate window, or leave empty
])

### para-Fluorofentanyl

*`known_emergences.csv`: 2021-01-01 to ongoing*

*polysubstance (trailing 4q, n=12): **adulterant/marker** -- alone 0%, named last 25%, with fentanyl 100%*

In [ ]:
plot_substance("para-Fluorofentanyl", spans=[
    ("2021-01-01", None),
])

## Part 2 -- appendix: all 211 substances, unannotated

Completeness/audit view, not a validation read: the ~176 substances not among Part 1's cells never alarmed under any of the five methods and are mostly flat or sparse lines with nothing to disagree about. No alarm markers here -- just the raw quarterly count, so a reviewer can confirm nothing in the long tail was overlooked, without 211 panels each carrying a legend.

Ordered by *current* EB05++TS score, descending -- same convention as Part 1, `geo export`'s one-hot columns, and `emergence_by_year.csv`'s rows.

In [ ]:
substances = quarterly.index.tolist()  # already current-EB05++TS-ordered
ncols = 14
nrows = -(-len(substances) // ncols)

fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 1.15, nrows * 0.75))
for ax, name in zip(axes.flat, substances):
    s = quarterly.loc[name]
    ax.plot(s.index, s.values, color="#2a78d6", linewidth=0.9)
    ax.fill_between(s.index, s.values, color="#2a78d6", alpha=0.12)
    ax.set_title(name, fontsize=5.5, pad=1.5)
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)
for ax in axes.flat[len(substances):]:
    ax.axis("off")

fig.suptitle("All 211 substances, raw quarterly deaths -- ordered by current "
            "EB05++TS score (audit view, no method annotation)",
            fontsize=10, y=1.005)
fig.tight_layout()
plt.show()